In [1]:
import json
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

BASE_DIR = r"D:\pycharm\Transformer"
DATA_DIR = os.path.join(BASE_DIR, "data", "Dataset")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(DATA_DIR, exist_ok=True)

print("数据目录:", DATA_DIR)

数据目录: D:\pycharm\Transformer\data\Dataset


In [2]:
import zipfile

zip_path = os.path.join(DATA_DIR, "Dataset.zip")

metadata_path = os.path.join(DATA_DIR, "metadata.json")


if not os.path.exists(metadata_path):

    print("开始解压数据...")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

    print("数据解压完成！")

else:
    print("检测到数据已存在，跳过解压。")

检测到数据已存在，跳过解压。


In [3]:
class SpeakerDataset(Dataset):

    def __init__(self, data_dir):


        with open(os.path.join(data_dir, "metadata.json"), "r") as f:
            metadata = json.load(f)

        self.data = []


        speakers = metadata["speakers"]


        self.speaker2id = {
            speaker: idx
            for idx, speaker in enumerate(speakers.keys())
        }


        for speaker, utts in speakers.items():


            for utt in utts:

                self.data.append({
                    "feature_path": utt["feature_path"],
                    "speaker_id": self.speaker2id[speaker]
                })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        item = self.data[idx]

        feat_path = os.path.join(
            DATA_DIR,
            item["feature_path"]
        )

        feat = torch.load(feat_path)

        label = item["speaker_id"]

        return feat, label

In [4]:
def collate_fn(batch):
    feats, labels = zip(*batch)

    lengths = [f.shape[0] for f in feats]
    max_len = max(lengths)

    padded = []
    for f in feats:
        pad_len = max_len - f.shape[0]
        pad = torch.zeros(pad_len, f.shape[1])
        padded.append(torch.cat([f, pad], dim=0))

    return torch.stack(padded), torch.tensor(labels)

In [5]:
class MediumModel(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.input_proj = nn.Linear(40, 256)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=8,
            dim_feedforward=512,
            dropout=0.2,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=4
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):

        x = self.input_proj(x)

        x = self.encoder(x)

        x = x.transpose(1, 2)

        x = self.pool(x).squeeze(-1)

        x = self.dropout(x)

        return self.fc(x)

In [6]:
dataset = SpeakerDataset(DATA_DIR)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

In [7]:
num_classes = len(dataset.speaker2id)

model = MediumModel(num_classes).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=25
)

EPOCHS = 25

In [8]:
for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    progress_bar = tqdm(
        loader,
        desc=f"Medium Epoch {epoch+1}/{EPOCHS}"
    )

    for x, y in progress_bar:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        pred = model(x)

        loss = criterion(pred, y)

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5
        )

        optimizer.step()

        total_loss += loss.item()

        progress_bar.set_postfix(
            loss=loss.item()
        )

    scheduler.step()

    avg_loss = total_loss / len(loader)

    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

Medium Epoch 1/25:   0%|          | 1/2170 [00:09<5:44:06,  9.52s/it, loss=6.58]


KeyboardInterrupt: 